# Lección 15.6 — Explicabilidad y regulación

> **Módulo 15 · ML en producción y responsabilidad** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 7.6 (importancia de variables, SHAP y dependencia parcial), Lecciones 15.4 y 15.5

### 🎯 Objetivos
Al terminar esta lección podrás:
- Distinguir explicaciones **globales** (cómo decide el modelo en general) y **locales** (por qué decidió esto para esta persona).
- Calcular e interpretar la **importancia por permutación** y la **dependencia parcial** con scikit-learn, y conocer sus trampas.
- Generar **razones de rechazo** y **contrafactuales** ("¿qué tendría que cambiar para que me aprueben?").
- Ubicar un sistema de IA en el marco regulatorio (Ley de IA de la UE, protección de datos) y redactar una **ficha del modelo** (*model card*).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance, partial_dependence
from sklearn.metrics import roc_auc_score

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el médico que explica 🩺

Imagina dos médicos igual de acertados. Uno te dice: *"Tienes que operarte"*, y nada más. El otro: *"Tienes que operarte porque tus análisis muestran X y la radiografía muestra Y; si Y no estuviera, bastaría con medicamentos"*.

Confías más en el segundo, puedes pedir una segunda opinión sobre algo concreto y, si se equivocó, puedes descubrir **dónde**. Eso es la **explicabilidad**: poder entender **por qué** un modelo decide lo que decide.

Y no es solo una cuestión de confianza: en muchos países, si un algoritmo te niega un crédito o un empleo, **tienes derecho a saber por qué**.

### ✋ Ejercicio 1 — Calentamiento
¿Para quién es útil una explicación en cada caso? Piensa en al menos tres "públicos" distintos para la explicación de un modelo de crédito.

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **El cliente:** ¿por qué me rechazaron y qué puedo hacer para que me aprueben?
- **El equipo de ciencia de datos:** ¿el modelo usa algo absurdo (fugas de datos, proxies, Lección 15.4)? ¿Por qué falla en estos casos?
- **El regulador / auditor:** ¿el modelo cumple la ley? ¿Discrimina?
- **El negocio:** ¿qué factores mueven el riesgo? ¿Confío en esto para decidir?

Cada público necesita un **tipo** distinto de explicación: global (cómo funciona en general) o local (este caso concreto).
</details>

---
## 2. 📖 El concepto: tipos de explicación

| | **Global**: ¿cómo decide en general? | **Local**: ¿por qué este caso? |
|---|---|---|
| **Modelos interpretables por diseño** (lineales, árboles pequeños) | Coeficientes, el árbol completo | Contribución de cada variable al caso; el camino en el árbol |
| **Métodos post-hoc** (para cualquier modelo, "caja negra") | Importancia por permutación, dependencia parcial (PDP) | SHAP, LIME, contrafactuales |

▶️ Trabajaremos con un modelo de crédito. Como los datos son **simulados**, conocemos la "verdad" y podremos comprobar si las explicaciones la recuperan:
- El riesgo baja con el **ingreso** y la **antigüedad laboral**.
- La **deuda** casi no importa hasta el **40 %** del ingreso; por encima, el riesgo se dispara.
- La **edad** y el **número de tarjetas** no influyen en nada.
- `ingreso_anual` es casi una copia de `ingreso_mensual` × 12 (los sistemas a veces guardan ambos).

In [ ]:
rng = np.random.default_rng(0)
n = 8_000
datos = pd.DataFrame({
    "ingreso_mensual":   rng.gamma(4, 5, n).round(1),            # miles de pesos
    "deuda_pct":         rng.uniform(0, 80, n).round(1),         # % del ingreso que va a deudas
    "antiguedad_anios":  rng.exponential(5, n).round(1),
    "edad":              rng.integers(20, 70, n).astype(float),    # float: lo pide partial_dependence
    "num_tarjetas":      rng.integers(0, 6, n).astype(float),
})
datos["ingreso_anual"] = (datos.ingreso_mensual * 12 + rng.normal(0, 5, n)).round(1)
riesgo = (-0.5 - 0.08 * datos.ingreso_mensual - 0.15 * datos.antiguedad_anios
          + 0.15 * np.maximum(datos.deuda_pct - 40, 0) + 0.01 * datos.deuda_pct)
datos["impago"] = (rng.random(n) < 1 / (1 + np.exp(-riesgo))).astype(int)

VARIABLES = ["ingreso_mensual", "ingreso_anual", "deuda_pct", "antiguedad_anios", "edad", "num_tarjetas"]
X_tr, X_te, y_tr, y_te = train_test_split(datos[VARIABLES], datos.impago, test_size=0.3, random_state=0)
modelo = HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr)
print(f"Tasa de impago: {datos.impago.mean():.1%} · AUC en prueba: {roc_auc_score(y_te, modelo.predict_proba(X_te)[:, 1]):.3f}")

### 📖 Importancia por permutación (repaso de la Lección 7.6)
Idea: **desordena** al azar una columna en el conjunto de prueba (rompiendo su relación con la etiqueta) y mide **cuánto empeora** el modelo. Si empeora mucho, el modelo dependía de esa variable.

### 🔮 Ejercicio 2 — ¿Qué variables importan?
Ordena de más a menos importante lo que **crees** que mostrará: `deuda_pct`, `ingreso_mensual`, `ingreso_anual`, `antiguedad_anios`, `edad`, `num_tarjetas`.

✍️ *Mi orden:*

In [ ]:
imp = permutation_importance(modelo, X_te, y_te, scoring="roc_auc", n_repeats=10, random_state=0)
importancias = pd.Series(imp.importances_mean, index=VARIABLES).sort_values()
importancias.plot.barh(xerr=pd.Series(imp.importances_std, index=VARIABLES)[importancias.index])
plt.xlabel("Caída del AUC al desordenar la variable"); plt.title("Importancia por permutación"); plt.show()
print(importancias.sort_values(ascending=False).round(4))

<details><summary><b>👀 Ver respuesta</b></summary>

`deuda_pct` es, de lejos, la más importante; luego la **antigüedad** y las variables de **ingreso**. `edad` y `num_tarjetas` quedan en ≈ 0: el modelo no las usa (bien, porque no influyen).

Pero fíjate en el **ingreso**: su importancia está **repartida** entre `ingreso_mensual` e `ingreso_anual`, y cada una por separado parece menos importante que la antigüedad. Es la gran trampa de la importancia por permutación, y la investigamos en la siguiente sección.
</details>

---
## 3. 📖 ¿Qué pasa si hay variables correlacionadas?

Cuando desordenas `ingreso_mensual`, el modelo todavía tiene `ingreso_anual` (casi la misma información), así que empeora **poco**. Y viceversa. Resultado: **dos variables importantes parecen poco importantes**.

### 🔮 Ejercicio 3 — Quita la copia
Si eliminamos `ingreso_anual` y reentrenamos, ¿qué pasará con la importancia de `ingreso_mensual`? ¿Y con el AUC?

✍️ *Mi predicción:*

In [ ]:
SIN_COPIA = [v for v in VARIABLES if v != "ingreso_anual"]
modelo2 = HistGradientBoostingClassifier(random_state=0).fit(X_tr[SIN_COPIA], y_tr)
print(f"AUC sin ingreso_anual: {roc_auc_score(y_te, modelo2.predict_proba(X_te[SIN_COPIA])[:, 1]):.3f}")
imp2 = permutation_importance(modelo2, X_te[SIN_COPIA], y_te, scoring="roc_auc", n_repeats=10, random_state=0)
print(pd.Series(imp2.importances_mean, index=SIN_COPIA).sort_values(ascending=False).round(4))

<details><summary><b>👀 Ver respuesta</b></summary>

El AUC prácticamente **no cambia** (la copia no aportaba información nueva) y la importancia de `ingreso_mensual` **sube mucho**: ahora recoge toda la importancia del ingreso, que antes estaba repartida.

Reglas prácticas:
- Antes de interpretar importancias, revisa las **correlaciones** entre variables (o agrúpalas: "ingreso" como un bloque).
- Una importancia baja **no significa** que la variable no tenga relación con la etiqueta; significa que **el modelo no la necesita** teniendo las demás.
</details>

### 📖 Dependencia parcial: ¿cómo influye una variable?
La importancia dice **cuánto** usa el modelo una variable, no **cómo**. La **dependencia parcial** (PDP) responde: *"si cambio solo esta variable para todos los clientes, ¿cómo cambia en promedio la predicción?"*.

### 🔮 Ejercicio 4 — La forma de la deuda
Dibuja en el aire la curva que esperas para el riesgo de impago según `deuda_pct` (de 0 a 80 %). Recuerda cómo simulamos los datos.

✍️ *Mi dibujo / descripción:*

In [ ]:
pd_deuda = partial_dependence(modelo2, X_te[SIN_COPIA], ["deuda_pct"], kind="average", response_method="predict_proba", method="brute")
pd_edad = partial_dependence(modelo2, X_te[SIN_COPIA], ["edad"], kind="average", response_method="predict_proba", method="brute")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
axes[0].plot(pd_deuda["grid_values"][0], pd_deuda["average"][0]); axes[0].set_xlabel("deuda_pct")
axes[1].plot(pd_edad["grid_values"][0], pd_edad["average"][0]); axes[1].set_xlabel("edad")
axes[0].set_ylabel("P(impago) promedio"); axes[0].axvline(40, ls=":", c="gray"); plt.show()

g, v = pd_deuda["grid_values"][0], pd_deuda["average"][0]
for x in [10, 30, 40, 50, 70]:
    print(f"deuda {x}% → P(impago) promedio ≈ {v[np.argmin(abs(g - x))]:.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

La curva de la deuda es **casi plana hasta el 40 %** y luego **sube con fuerza**: el modelo descubrió solo el "codo" que pusimos en la simulación (≈ 0.06–0.12 por debajo del 40 %, ≈ 0.31 en el 50 % y ≈ 0.89 en el 70 %). La curva de la edad es **plana**: la edad no cambia el riesgo.

Esto es muy útil para validar un modelo con expertos: un analista de crédito puede mirar la curva y decir "sí, tiene sentido" o "¡esto es absurdo!". Cuidado: la PDP promedia sobre todos los clientes y puede esconder efectos que van en direcciones opuestas en subgrupos (para eso existen las curvas ICE, `kind="individual"`).
</details>

---
## 4. 📖 Explicaciones locales: ¿por qué **me** rechazaron?

### Opción 1: un modelo interpretable por diseño
En una regresión logística, la contribución de cada variable a la decisión de un cliente es **coeficiente × (valor del cliente − valor promedio)**. Las contribuciones que más empujan hacia el impago son las **razones de rechazo**. (SHAP generaliza esta idea a cualquier modelo; en Colab: `pip install shap`.)

In [ ]:
logistica = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr[SIN_COPIA], y_tr)
print(f"AUC logística: {roc_auc_score(y_te, logistica.predict_proba(X_te[SIN_COPIA])[:, 1]):.3f} "
      f"(boosting: {roc_auc_score(y_te, modelo2.predict_proba(X_te[SIN_COPIA])[:, 1]):.3f})")

def razones(cliente):
    escalador, lr = logistica[0], logistica[-1]
    z = (cliente[SIN_COPIA].values - escalador.mean_) / escalador.scale_     # distancia al cliente promedio
    contrib = pd.Series(lr.coef_[0] * z, index=SIN_COPIA).sort_values(ascending=False)
    return contrib

clientes_ejemplo = X_te.iloc[[0, 1, 2]]
print(clientes_ejemplo[SIN_COPIA])
print("\nP(impago) según la logística:", logistica.predict_proba(clientes_ejemplo[SIN_COPIA])[:, 1].round(2))

### ✋ Ejercicio 5 — Lee las razones
Elige al cliente con mayor probabilidad de impago de la tabla de arriba y ejecuta la celda con su posición (0, 1 o 2). ¿Cuáles son sus **dos principales razones** de rechazo? ¿Tienen sentido mirando sus datos?

✍️ *Mis razones:*

In [ ]:
POSICION = 0      # ✏️ cambia a 1 o 2
r = razones(clientes_ejemplo.iloc[POSICION])
print("Contribuciones (positivas = empujan hacia el impago):")
print(r.round(2))
print("\nPrincipales razones de rechazo:", list(r.index[:2]))

<details><summary><b>👀 Ver respuesta</b></summary>

Las razones dependen del cliente: suele aparecer la **deuda alta** cuando supera el 40 %, y luego un **ingreso bajo** o una **antigüedad corta**. Compruébalo con sus datos: la variable con mayor contribución positiva debería estar claramente "peor" que el promedio (más deuda, menos ingreso o menos antigüedad).

Fíjate en el AUC: ¡la logística (≈ 0.906) rinde **igual o un poco mejor** que el boosting (≈ 0.903)! Aunque una recta no dibuja exactamente el "codo" del 40 %, en estos datos alcanza para ordenar bien a los clientes. Muchas veces se supone un **compromiso entre interpretabilidad y rendimiento** que, al medirlo, no existe. Cynthia Rudin (2019) defiende justamente eso: en decisiones de alto riesgo, prueba primero un modelo interpretable bien diseñado (por ejemplo, añadiendo la variable "deuda por encima del 40 %") antes de explicar una caja negra a posteriori.
</details>

### Opción 2: contrafactuales — "¿qué tendría que cambiar?"
Para el cliente, la explicación más útil suele ser accionable: *"Si tu deuda fuera del 35 % en lugar del 55 %, te aprobaríamos"*. Funciona con **cualquier** modelo: basta probar cambios y ver cuándo cambia la decisión.

### 🔮 Ejercicio 6 — El contrafactual
Buscaremos un cliente rechazado (P(impago) > 0.5 según el boosting) y calcularemos: (a) cuánto tendría que bajar su deuda, y (b) cuánto tendría que subir su ingreso para ser aprobado. ¿Cuál crees que será más "realista"?

✍️ *Mi predicción:*

In [ ]:
prob = modelo2.predict_proba(X_te[SIN_COPIA])[:, 1]
rechazado = X_te[SIN_COPIA][prob > 0.5].iloc[0]
print("Cliente rechazado:", rechazado.to_dict(), f"· P(impago) = {modelo2.predict_proba(rechazado.to_frame().T)[0, 1]:.2f}")

def contrafactual(cliente, variable, valores):
    for v in valores:
        c = cliente.copy(); c[variable] = v
        if modelo2.predict_proba(c.to_frame().T)[0, 1] < 0.5:
            return v
    return None

deuda_ok = contrafactual(rechazado, "deuda_pct", np.arange(rechazado.deuda_pct, -0.1, -0.5))
ingreso_ok = contrafactual(rechazado, "ingreso_mensual", np.arange(rechazado.ingreso_mensual, 200, 0.5))
print(f"(a) Aprobado si su deuda baja de {rechazado.deuda_pct}% a {deuda_ok:.1f}%")
print(f"(b) Aprobado si su ingreso sube de {rechazado.ingreso_mensual} a {ingreso_ok} miles" if ingreso_ok else
      "(b) Ningún ingreso razonable alcanza para aprobarlo")

<details><summary><b>👀 Ver respuesta</b></summary>

Este cliente tiene una deuda del ≈ 80 %. Lo realista es **bajar la deuda**: con ≈ 54 % ya sería aprobado. En cambio, **ningún** ingreso razonable alcanza: con una deuda tan alta, el riesgo lo domina la deuda (la curva de la Sección 3).

Los contrafactuales son potentes, pero cuidado:
- Deben ser **accionables** ("baja tu deuda") y no absurdos ("ten 10 años menos").
- Si el contrafactual dice "múdate de código postal", ¡has descubierto un **proxy**! (Lección 15.4).
- Un modelo con varias variables puede tener **muchos** contrafactuales; se suele mostrar el más cercano o el más fácil.
</details>

---
## 5. 📖 Las trampas de la explicabilidad

| Trampa | Qué pasa | Cómo evitarla |
|---|---|---|
| **Importancia ≠ causalidad** | "El modelo usa mucho X" no significa que **cambiar X cambie la realidad** | No prometer efectos causales; experimentos (A/B) para eso |
| **Variables correlacionadas** | La importancia se reparte o se esconde (Sección 3) | Revisar correlaciones, agrupar variables |
| **Explicaciones inestables** | LIME/SHAP pueden cambiar con pequeñas variaciones | Varias semillas, comparar métodos |
| **Explicaciones "a la medida"** | Se puede construir un modelo que **parezca** justo en sus explicaciones y no lo sea | Auditar el comportamiento (Lección 15.4), no solo las explicaciones |
| **Explicar el modelo ≠ explicar el mundo** | Si el modelo está mal, su explicación también describe algo equivocado | Validar con expertos y con datos |

### ✋ Ejercicio 7 — Importancia y causalidad
Un modelo de ventas muestra que la variable más importante es "número de paraguas vendidos ayer". El gerente propone **regalar paraguas** para subir las ventas. ¿Qué le respondes?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Que la importancia dice que la variable **ayuda a predecir**, no que la **cause**. Probablemente los paraguas son una señal de **lluvia**, y la lluvia (gente en centros comerciales, pedidos a domicilio…) es lo que mueve las ventas. Regalar paraguas no hará llover. Para saber si una acción causa un efecto hace falta un **experimento** (Lección 14.2: pruebas A/B) o métodos de inferencia causal.
</details>

---
## 6. 📖 Regulación: el marco que ya está aquí

> ⚖️ *Esto no es asesoría legal y las leyes cambian: verifica siempre el estado vigente en tu país.*

| Norma | Dónde | Idea clave para quien hace ML |
|---|---|---|
| **Ley de IA de la UE** (*AI Act*, 2024) | Unión Europea (y quien venda allí) | Clasifica los sistemas por **riesgo**. Prohíbe algunos usos; los de **alto riesgo** necesitan gestión de riesgos, datos de calidad, documentación, registros, supervisión humana, exactitud y robustez. Aplicación escalonada entre 2025 y 2027 |
| **RGPD**, art. 22 y transparencia | Unión Europea | Derecho a no ser objeto de decisiones **solo automatizadas** con efectos significativos, salvo excepciones, y a recibir información significativa sobre la lógica aplicada |
| Leyes de protección de datos de América Latina (p. ej., **LGPD** de Brasil, art. 20) | Brasil y otros países | Derecho a pedir **revisión** de decisiones automatizadas; varios países discuten leyes específicas de IA |
| Leyes de crédito justo (p. ej., **ECOA** en EE. UU.) | EE. UU. | Al negar un crédito hay que dar las **razones principales** (*adverse action notice*), también si decide un modelo |
| **Ley Local 144** de Nueva York (2023) | Ciudad de Nueva York | Las herramientas automatizadas de contratación necesitan una **auditoría de sesgo** anual (Lección 15.4) |

### 📖 Los niveles de riesgo de la Ley de IA de la UE

| Nivel | Ejemplos | Qué implica |
|---|---|---|
| 🚫 **Prohibido** | Puntuación social de ciudadanos por gobiernos, manipulación que explota vulnerabilidades, ciertos usos de reconocimiento biométrico en tiempo real | No se puede usar |
| 🔴 **Alto riesgo** | Selección de personal, crédito, educación (calificar exámenes), acceso a servicios públicos, dispositivos médicos, infraestructuras críticas | Obligaciones estrictas (documentación, supervisión humana, gestión de riesgos…) |
| 🟡 **Riesgo limitado / transparencia** | Chatbots, contenido generado por IA (*deepfakes*) | Avisar a las personas de que interactúan con una IA o de que el contenido es sintético |
| 🟢 **Riesgo mínimo** | Filtros de spam, recomendaciones en videojuegos | Sin obligaciones específicas |

### ✋ Ejercicio 8 — Clasifica el riesgo
¿En qué nivel crees que caería cada sistema?
1. El filtro de CVs de la Lección 15.4.
2. Un chatbot de atención al cliente de una aerolínea.
3. El modelo de cancelación de clientes de la Lección 15.1 (para priorizar llamadas de retención).
4. Un sistema que puntúa a los ciudadanos según su comportamiento en redes para dar o quitar acceso a servicios públicos.
5. El modelo de crédito de esta lección.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. 🔴 **Alto riesgo** (empleo y selección de personal).
2. 🟡 **Transparencia**: debe quedar claro que es una IA.
3. 🟢 **Mínimo**, en principio: priorizar llamadas comerciales no afecta derechos fundamentales (aunque si se usara para negar servicios o subir precios, habría que revisarlo).
4. 🚫 **Prohibido** (puntuación social).
5. 🔴 **Alto riesgo** (evaluación de solvencia de personas físicas).

En la práctica, la clasificación exacta la determina el texto de la ley y su interpretación jurídica; lo importante es **preguntárselo desde el diseño**, no después de desplegar.
</details>

### 📖 La ficha del modelo (*model card*)
Propuesta por Mitchell et al. (Google, 2019), es un documento corto que acompaña a cada modelo y responde: qué hace, para qué sirve (y para qué **no**), con qué datos se entrenó, cómo se evaluó (también **por grupos**), sus limitaciones y riesgos. Para los datos existe la idea hermana: las *datasheets for datasets* (Gebru et al., 2018). Es exactamente la documentación que piden los reguladores… y la que ya empezaste a escribir en la Lección 15.1.

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Una variable "sin sentido" aparece como la más importante | Fuga de datos o proxy | Revisar cómo se creó la variable (Lecciones 2.6, 15.4) |
| Dos variables relacionadas aparecen poco importantes | Importancia repartida por correlación | Quitar o agrupar una y recalcular |
| La explicación de un caso cambia mucho con un cambio mínimo | Modelo inestable o método de explicación inestable | Revisar con varios métodos y semillas |
| El cliente recibe "razones" que no puede cambiar ni entender | Explicaciones no accionables | Contrafactuales con variables accionables y lenguaje claro |
| Nadie sabe qué datos ni qué versión decidieron un caso | Falta de registro (Lección 15.3) | Ficha del modelo + registro de decisiones |

### 🔮 Ejercicio 9 — Diagnóstico
En un modelo que predice si un paciente será readmitido en el hospital, la variable más importante es `numero_de_habitacion`. ¿Qué sospechas?

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

Una habitación no causa readmisiones: probablemente es un **proxy** de algo (las habitaciones de cierta planta son de cuidados intensivos o de cierta especialidad) o una **fuga de datos** (por ejemplo, se reasigna a los pacientes que ya se sabe que volverán a cierta habitación). Hay que investigar cómo se genera esa variable antes de confiar en el modelo. ¡Para eso sirve la explicabilidad global!
</details>

---
## 8. 📋 Ficha: ¿qué método de explicación uso?

| Pregunta | Método | Herramienta |
|---|---|---|
| ¿Qué variables usa el modelo en general? | Importancia por permutación | `sklearn.inspection.permutation_importance` |
| ¿Cómo influye una variable (forma de la relación)? | Dependencia parcial / ICE | `sklearn.inspection.partial_dependence`, `PartialDependenceDisplay` |
| ¿Por qué esta predicción? | Contribuciones (lineal), SHAP, LIME | coeficientes; `shap`, `lime` (para Colab) |
| ¿Qué tendría que cambiar el cliente? | Contrafactuales | búsqueda simple; `dice-ml` (para Colab) |
| ¿Cómo documento todo? | Ficha del modelo | *model cards*, *datasheets* |

## 🧠 Resumen en 5 puntos
1. Hay explicaciones **globales** (cómo decide en general) y **locales** (por qué este caso); y modelos **interpretables por diseño** vs. métodos **post-hoc**.
2. La **importancia por permutación** mide cuánto depende el modelo de cada variable; con variables **correlacionadas** se reparte y engaña.
3. La **dependencia parcial** muestra **cómo** influye una variable; los **contrafactuales** dicen qué tendría que cambiar para otra decisión.
4. **Importancia no es causalidad**, y una explicación no reemplaza una auditoría del comportamiento.
5. La regulación (Ley de IA de la UE, protección de datos, crédito justo) exige cada vez más **documentación, explicaciones y supervisión humana** en sistemas de alto riesgo.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** "¿Por qué el modelo rechazó a **esta** persona?" pide una explicación…
- a) Global · b) Local · c) Causal · d) Estadística

**2.** Desordenas una variable y el AUC no cambia. Lo más correcto es concluir que…
- a) La variable no tiene ninguna relación con la etiqueta · b) El modelo no la necesita, quizás porque otra variable lleva la misma información · c) Hay fuga de datos · d) El modelo está mal

**3.** La dependencia parcial de "deuda" es plana hasta el 40 % y luego sube. Significa que…
- a) La deuda no importa · b) En promedio, el modelo solo aumenta el riesgo cuando la deuda supera ~40 % · c) Hay que quitar la deuda · d) El modelo es lineal

**4.** Según la Ley de IA de la UE, un sistema para seleccionar candidatos a un empleo es…
- a) De riesgo mínimo · b) De alto riesgo · c) Prohibido · d) No regulado

**5.** ¿Cuál es un buen contrafactual para un cliente?
- a) "Si fueras 10 años más joven, te aprobaríamos" · b) "Si tu deuda bajara del 55 % al 40 %, te aprobaríamos" · c) "El modelo es complejo" · d) "Tu puntaje es 0.62"

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** 2. **b)** 3. **b)** 4. **b)** 5. **b)** (accionable y comprensible; la a) no es accionable y además usa la edad, una variable protegida)
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Un banco tiene dos opciones: un boosting con AUC 0.86 que explica con SHAP, o una logística bien diseñada con AUC 0.84 cuyas razones salen directamente de los coeficientes. ¿Qué preguntas harías antes de elegir? (Conecta con la Lección 5.4, Ejercicio 9.2 B.)

**B. Caso de análisis 💼** Una universidad usa un modelo para decidir qué estudiantes reciben una beca. Un estudiante rechazado pide una explicación. Escribe la respuesta que le darías (3–4 frases), sin jerga técnica.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** ¿Cuánto vale en dinero (o en personas mal atendidas) la diferencia de 0.02 de AUC? ¿Qué exige el regulador: le basta una explicación post-hoc o pide un modelo interpretable? ¿Las explicaciones SHAP son estables? ¿Ambos modelos son igual de justos por grupo (Lección 15.4)? ¿Quién mantendrá y auditará cada uno? En decisiones de alto riesgo, muchas veces gana el interpretable si la pérdida de rendimiento es pequeña.

**B.** Un ejemplo:

> «Tu solicitud fue evaluada con un sistema que considera tu promedio, la situación económica de tu familia y tu participación en actividades. En tu caso, el factor que más pesó fue el promedio de los dos últimos semestres, que quedó por debajo del de los becados de este año; con un promedio de 8.5 o más, tu solicitud habría sido aprobada. Puedes pedir que una persona del comité revise tu caso, por ejemplo si hubo circunstancias que el sistema no conoce. Para hacerlo, escribe a ...»
</details>

### 9.3 🧩 Práctica integradora: "La ficha del modelo de crédito"

Vas a preparar la documentación del modelo de crédito (`modelo2`) para el comité de riesgos y el regulador, en 4 pasos.

**Paso 1 ▶️ — Desempeño general y por grupo de edad.** La edad es una característica protegida en muchos contextos. El modelo no la necesita (Sección 2), pero la recibe. ¿Trata distinto a los grupos de edad?

In [ ]:
ev = X_te[SIN_COPIA].copy()
ev["impago"] = y_te.values
ev["prob"] = modelo2.predict_proba(X_te[SIN_COPIA])[:, 1]
ev["rechazado"] = (ev.prob > 0.5).astype(int)
ev["grupo_edad"] = pd.cut(ev.edad, [19, 30, 50, 70], labels=["20–30", "31–50", "51–69"])

por_grupo = ev.groupby("grupo_edad", observed=True).apply(lambda d: pd.Series({
    "n": len(d), "tasa de rechazo": d.rechazado.mean(), "tasa real de impago": d.impago.mean(),
    "AUC": roc_auc_score(d.impago, d.prob)}), include_groups=False)
print(f"AUC global: {roc_auc_score(ev.impago, ev.prob):.3f}")
por_grupo.round(3)

✋ ¿Hay diferencias relevantes entre grupos de edad?

✍️ *Mi lectura:*

<details><summary><b>👀 Ver respuesta</b></summary>

Las tasas de rechazo, las tasas reales de impago y los AUC son **parecidos** en los tres grupos (las pequeñas diferencias son ruido de muestreo con entre 500 y 1,000 clientes por grupo). Era de esperar: la edad no influye en el riesgo simulado y el modelo casi no la usa. Aun así, **había que comprobarlo y documentarlo**: es lo que pedirá el regulador.
</details>

**Paso 2 ✏️ — ¿Y si quitamos la edad del modelo?** Reentrena sin `edad` y compara el AUC. Si no pierde nada, ¿conviene quitarla?

In [ ]:
SIN_EDAD = [v for v in SIN_COPIA if v != "edad"]     # ✏️
modelo3 = HistGradientBoostingClassifier(random_state=0).fit(X_tr[SIN_EDAD], y_tr)
print(f"AUC con edad: {roc_auc_score(y_te, modelo2.predict_proba(X_te[SIN_COPIA])[:, 1]):.3f} · "
      f"sin edad: {roc_auc_score(y_te, modelo3.predict_proba(X_te[SIN_EDAD])[:, 1]):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El AUC es prácticamente igual. Conviene **quitarla** (y también `num_tarjetas`, que tampoco aporta): menos variables significa menos riesgo legal, menos datos personales que recolectar (principio de **minimización de datos**) y menos cosas que monitorear. Pero sigue guardando la edad **aparte** para poder **auditar** por grupos, como en el Paso 1.
</details>

**Paso 3 ▶️ — Razones de rechazo para 3 clientes.** Usando el modelo final (`modelo3`) y contrafactuales sobre la deuda, genera un mensaje por cliente rechazado.

In [ ]:
rechazados = X_te[SIN_EDAD][modelo3.predict_proba(X_te[SIN_EDAD])[:, 1] > 0.5].head(3)
for _, c in rechazados.iterrows():
    meta = None
    for v in np.arange(c.deuda_pct, -0.1, -0.5):
        cc = c.copy(); cc["deuda_pct"] = v
        if modelo3.predict_proba(cc.to_frame().T)[0, 1] < 0.5:
            meta = v; break
    if meta is not None:
        print(f"• Deuda {c.deuda_pct:.0f}% → «Su solicitud sería aprobada si sus pagos de deuda bajaran al {meta:.0f}% de su ingreso.»")
    else:
        print(f"• Deuda {c.deuda_pct:.0f}% → «Reducir la deuda no basta; los factores principales son su ingreso y su antigüedad laboral.»")

<details><summary><b>👀 Ver respuesta</b></summary>

A los tres les basta con bajar la deuda, pero la meta **depende de cada persona** (≈ 49 %, ≈ 59 % y ≈ 42 %): quien tiene más ingreso o antigüedad tolera más deuda. Por eso un mensaje genérico ("baje su deuda al 40 %") sería injusto con unos y engañoso con otros. Si para algún cliente bajar la deuda no bastara, el mensaje señalaría el ingreso o la antigüedad. Son explicaciones **accionables** y en lenguaje claro: lo que exigen leyes como la ECOA y lo que agradece cualquier cliente.
</details>

**Paso 4 ✋ — Escribe la ficha del modelo.** Completa esta plantilla (una página máximo) con lo que aprendiste en el módulo:

```
FICHA DEL MODELO: Riesgo de impago v1
1. Uso previsto: ...            Usos NO previstos: ...
2. Datos: ...                   Variables: ...  (y cuáles se excluyeron y por qué)
3. Desempeño: AUC global ___ ; por grupo de edad: ...
4. Cómo explica sus decisiones: ...
5. Limitaciones y riesgos: ...
6. Monitoreo y reentrenamiento (15.2–15.3): ...
7. Supervisión humana: ...
```

✍️ *Mi ficha:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:

> **1. Uso previsto:** estimar la probabilidad de impago de solicitudes de crédito personal para apoyar (no reemplazar) la decisión del analista. **No previsto:** fijar tasas de interés, decisiones de empleo o cobranza.
> **2. Datos:** 8,000 solicitudes históricas (simuladas). Variables: ingreso mensual, % de deuda, antigüedad laboral. Excluidas: ingreso anual (redundante), edad y número de tarjetas (no aportan; la edad se conserva aparte solo para auditar).
> **3. Desempeño:** AUC ≈ 0.90 en prueba; tasas de rechazo y AUC similares en los grupos de edad 20–30, 31–50 y 51–69.
> **4. Explicaciones:** importancia por permutación y dependencia parcial para el comité (la deuda por encima del 40 % es el factor dominante); contrafactuales accionables para cada cliente rechazado.
> **5. Limitaciones:** datos de un solo período; no evaluado en clientes sin historial laboral; sensibilidad a cambios económicos (inflación, tasas).
> **6. Monitoreo:** PSI mensual de las variables y de las predicciones; AUC y tasa de impago por cohorte cuando lleguen las etiquetas; reentrenamiento si el AUC cae más de 0.03 (campeón vs. retador).
> **7. Supervisión humana:** todo rechazo puede ser revisado por un analista a petición del cliente; las solicitudes con probabilidad entre 0.4 y 0.6 siempre pasan por revisión humana.
</details>

---
## 📚 Para profundizar
- 📘 **Libro (gratis):** Christoph Molnar, *Interpretable Machine Learning*.
- 📰 **Artículos:** Rudin (2019), *Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead* · Mitchell et al. (2019), *Model Cards for Model Reporting* · Wachter et al. (2017), *Counterfactual Explanations without Opening the Black Box*.
- ⚖️ **Regulación:** texto y resúmenes oficiales de la Ley de IA de la UE (*AI Act*); guías de las autoridades de protección de datos de tu país.
- 🛠️ **Para Colab:** `shap`, `lime`, `dice-ml` (contrafactuales), `interpret` (modelos interpretables de Microsoft, como EBM).

---
⬅️ **Anterior:** [15.5 Privacidad, seguridad y ataques adversarios](15.5-privacidad-seguridad-y-ataques-adversarios.ipynb) · ➡️ **Siguiente:** [15.7 Proyecto del módulo: Auditoría de un modelo de crédito en producción 🏦🔍](15.7-proyecto-del-modulo.md)